# Data Cleaning

This notebook loads the same dataset used in the exploration notebook, cleans it, and saves a clean CSV file for the next project steps.

In [1]:
import re
from pathlib import Path
from io import StringIO

import pandas as pd
import requests

DATA_URL = "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset-v2/CTU-IoT-Malware-Capture-34-1/2018-12-21-15-50-14-192.168.1.195-zeek-conn-log.labeled"

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "data" / "cleaned"
OUTPUT_PATH = OUTPUT_DIR / "iot23_cleaned.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output file:", OUTPUT_PATH)

Output file: C:\Users\Mayar\OneDrive\Desktop\University\grad_project\IoT_GNN\data\cleaned\iot23_cleaned.csv


## Load dataset

In [2]:
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()

raw_text = response.text
print("Downloaded size:", round(len(response.content) / (1024 * 1024), 2), "MB")

Downloaded size: 2.88 MB


In [3]:
metadata_lines = [line for line in raw_text.splitlines() if line.startswith("#")]
data_lines = [line for line in raw_text.splitlines() if line.strip() and not line.startswith("#")]

fields_line = next(line for line in metadata_lines if line.startswith("#fields"))
columns = re.split(r"\t|\s{3,}", fields_line.replace("#fields\t", ""))

df = pd.read_csv(
    StringIO("\n".join(data_lines)),
    sep=r"\t|\s{3,}",
    engine="python",
    header=None,
    names=columns,
    na_values=["-", "(empty)", ""],
)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
df.head()

Rows: 23145
Columns: 23


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed-label
0,1.545404e+09,CrDn63WjJEmrWGjqf,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,3.139211,0.0,...,NaN,0,S,3,180,0,0,NaN,Benign,NaN
1,1.545404e+09,CY9lJW3gh1Eje4usP6,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,NaN,0,S,1,60,0,0,NaN,Benign,NaN
2,1.545404e+09,CcFXLynukEDnUlvgl,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,NaN,0,S,1,60,0,0,NaN,Benign,NaN
3,1.545404e+09,CDrkrSobGYxHhYfth,192.168.1.195,41040,185.244.25.235,80,tcp,http,1.477656,149.0,...,NaN,2896,ShADadttcfF,94,5525,96,139044,NaN,Benign,NaN
4,1.545404e+09,CTWZQf2oJSvq6zmPAc,192.168.1.195,41042,185.244.25.235,80,tcp,NaN,3.147116,0.0,...,NaN,0,S,3,180,0,0,NaN,Benign,NaN


## Clean column names and values

In [4]:
df_clean = df.copy()

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
    .str.replace("-", "_", regex=False)
)

text_columns = df_clean.select_dtypes(include="object").columns
for column in text_columns:
    df_clean[column] = df_clean[column].str.strip()

df_clean.head()

C:\Users\Mayar\AppData\Local\Temp\ipykernel_15020\2713859712.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df_clean.select_dtypes(include="object").columns


,ts,uid,id_orig_h,id_orig_p,id_resp_h,id_resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed_label
0,1.545404e+09,CrDn63WjJEmrWGjqf,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,3.139211,0.0,...,NaN,0,S,3,180,0,0,NaN,Benign,NaN
1,1.545404e+09,CY9lJW3gh1Eje4usP6,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,NaN,0,S,1,60,0,0,NaN,Benign,NaN
2,1.545404e+09,CcFXLynukEDnUlvgl,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,NaN,0,S,1,60,0,0,NaN,Benign,NaN
3,1.545404e+09,CDrkrSobGYxHhYfth,192.168.1.195,41040,185.244.25.235,80,tcp,http,1.477656,149.0,...,NaN,2896,ShADadttcfF,94,5525,96,139044,NaN,Benign,NaN
4,1.545404e+09,CTWZQf2oJSvq6zmPAc,192.168.1.195,41042,185.244.25.235,80,tcp,NaN,3.147116,0.0,...,NaN,0,S,3,180,0,0,NaN,Benign,NaN


## Fix data types

In [5]:
numeric_columns = [
    "ts",
    "id_orig_p",
    "id_resp_p",
    "duration",
    "orig_bytes",
    "resp_bytes",
    "missed_bytes",
    "orig_pkts",
    "orig_ip_bytes",
    "resp_pkts",
    "resp_ip_bytes",
]

for column in numeric_columns:
    if column in df_clean.columns:
        df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

df_clean.dtypes

ts                float64
uid                   str
id_orig_h             str
id_orig_p           int64
id_resp_h             str
id_resp_p           int64
proto                 str
service               str
duration          float64
orig_bytes        float64
resp_bytes        float64
conn_state            str
local_orig        float64
local_resp        float64
missed_bytes        int64
history               str
orig_pkts           int64
orig_ip_bytes       int64
resp_pkts           int64
resp_ip_bytes       int64
tunnel_parents    float64
label                 str
detailed_label        str
dtype: object

## Remove duplicates and unusable rows

In [6]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
duplicates_removed = rows_before - len(df_clean)

required_columns = ["ts", "uid", "id_orig_h", "id_resp_h", "proto", "label"]
available_required_columns = [column for column in required_columns if column in df_clean.columns]

rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=available_required_columns).reset_index(drop=True)
missing_required_removed = rows_before - len(df_clean)

print("Duplicates removed:", duplicates_removed)
print("Rows removed because important values were missing:", missing_required_removed)
print("Cleaned rows:", df_clean.shape[0])
print("Cleaned columns:", df_clean.shape[1])

Duplicates removed: 0
Rows removed because important values were missing: 0
Cleaned rows: 23145
Cleaned columns: 23


## Prepare labels

In [7]:
df_clean["label"] = df_clean["label"].str.lower()

if "detailed_label" in df_clean.columns:
    df_clean["detailed_label"] = df_clean["detailed_label"].fillna("unknown").str.lower()

df_clean["target"] = (df_clean["label"] != "benign").astype(int)

print("Label counts:")
display(df_clean["label"].value_counts())

print("Target counts: 0 = benign, 1 = attack")
display(df_clean["target"].value_counts().sort_index())

Label counts:


label
malicious    21222
benign        1923
Name: count, dtype: int64

Target counts: 0 = benign, 1 = attack


target
0     1923
1    21222
Name: count, dtype: int64

## Missing values report

In [8]:
missing_report = pd.DataFrame(
    {
        "missing_count": df_clean.isna().sum(),
        "missing_percent": (df_clean.isna().mean() * 100).round(2),
        "data_type": df_clean.dtypes.astype(str),
    }
).sort_values("missing_count", ascending=False)

missing_report

,missing_count,missing_percent,data_type
local_orig,23145,100.00,float64
tunnel_parents,23145,100.00,float64
local_resp,23145,100.00,float64
service,21298,92.02,str
duration,17824,77.01,float64
resp_bytes,17824,77.01,float64
orig_bytes,17824,77.01,float64
ts,0,0.00,float64
proto,0,0.00,str
id_resp_p,0,0.00,int64


## Save cleaned dataset

In [9]:
df_clean.to_csv(OUTPUT_PATH, index=False)

print("Saved cleaned dataset to:", OUTPUT_PATH)
print("Final shape:", df_clean.shape)
df_clean.head()

Saved cleaned dataset to: C:\Users\Mayar\OneDrive\Desktop\University\grad_project\IoT_GNN\data\cleaned\iot23_cleaned.csv
Final shape: (23145, 24)


,ts,uid,id_orig_h,id_orig_p,id_resp_h,id_resp_p,proto,service,duration,orig_bytes,...,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed_label,target
0,1.545404e+09,CrDn63WjJEmrWGjqf,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,3.139211,0.0,...,0,S,3,180,0,0,NaN,benign,unknown,0
1,1.545404e+09,CY9lJW3gh1Eje4usP6,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,0,S,1,60,0,0,NaN,benign,unknown,0
2,1.545404e+09,CcFXLynukEDnUlvgl,192.168.1.195,41040,185.244.25.235,80,tcp,NaN,NaN,NaN,...,0,S,1,60,0,0,NaN,benign,unknown,0
3,1.545404e+09,CDrkrSobGYxHhYfth,192.168.1.195,41040,185.244.25.235,80,tcp,http,1.477656,149.0,...,2896,ShADadttcfF,94,5525,96,139044,NaN,benign,unknown,0
4,1.545404e+09,CTWZQf2oJSvq6zmPAc,192.168.1.195,41042,185.244.25.235,80,tcp,NaN,3.147116,0.0,...,0,S,3,180,0,0,NaN,benign,unknown,0
